In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/SVAMPsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CCoT_prompt_example.txt").read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        prompt_q = (
            CoT_prompt_examples +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accurately."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Answer
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        # === Determine Correctness
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:03<10:21,  3.05s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/205 [00:03<02:19,  1.44it/s]

Accuracy: 3 / 4 = 75.00%
Accuracy: 4 / 5 = 80.00%
Accuracy: 5 / 6 = 83.33%
Accuracy: 6 / 7 = 85.71%
Accuracy: 7 / 8 = 87.50%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%


  6%|▌         | 12/205 [00:03<00:36,  5.27it/s]

...

Accuracy: 174 / 193 = 90.16%
Accuracy: 175 / 194 = 90.21%
Accuracy: 176 / 195 = 90.26%
Accuracy: 177 / 196 = 90.31%
Accuracy: 178 / 197 = 90.36%
Accuracy: 179 / 198 = 90.40%
Accuracy: 180 / 199 = 90.45%


 98%|█████████▊| 200/205 [08:06<00:09,  1.96s/it]

Accuracy: 181 / 200 = 90.50%


100%|██████████| 205/205 [08:07<00:00,  2.38s/it]

Accuracy: 182 / 201 = 90.55%
Accuracy: 183 / 202 = 90.59%
Accuracy: 184 / 203 = 90.64%
Accuracy: 184 / 204 = 90.20%
Accuracy: 185 / 205 = 90.24%


In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
results = []
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                global acc
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n") 

  0%|          | 0/205 [00:00<?, ?it/s]

  0%|          | 1/205 [00:01<04:25,  1.30s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/205 [00:01<02:34,  1.31it/s]

Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


...

Accuracy: 143 / 162 = 88.27%
Accuracy: 143 / 163 = 87.73%
Accuracy: 144 / 164 = 87.80%
Accuracy: 145 / 165 = 87.88%
Accuracy: 146 / 166 = 87.95%
Accuracy: 147 / 167 = 88.02%
Accuracy: 148 / 168 = 88.10%
Accuracy: 149 / 169 = 88.17%
Accuracy: 150 / 170 = 88.24%
Accuracy: 151 / 171 = 88.30%
Accuracy: 152 / 172 = 88.37%
Accuracy: 153 / 173 = 88.44%
Accuracy: 154 / 174 = 88.51%
Accuracy: 155 / 175 = 88.57%
Accuracy: 156 / 176 = 88.64%
Accuracy: 157 / 177 = 88.70%
Accuracy: 158 / 178 = 88.76%
Accuracy: 159 / 179 = 88.83%
Accuracy: 160 / 180 = 88.89%
Accuracy: 161 / 181 = 88.95%
Accuracy: 162 / 182 = 89.01%


 89%|████████▉ | 183/205 [07:03<00:37,  1.73s/it]

Accuracy: 163 / 183 = 89.07%
Accuracy: 164 / 184 = 89.13%
Accuracy: 165 / 185 = 89.19%
Accuracy: 166 / 186 = 89.25%
Accuracy: 167 / 187 = 89.30%
Accuracy: 168 / 188 = 89.36%


100%|██████████| 205/205 [08:00<00:00,  2.34s/it]

Accuracy: 169 / 189 = 89.42%
Accuracy: 170 / 190 = 89.47%
Accuracy: 171 / 191 = 89.53%
Accuracy: 172 / 192 = 89.58%
Accuracy: 173 / 193 = 89.64%
Accuracy: 174 / 194 = 89.69%
Accuracy: 175 / 195 = 89.74%
Accuracy: 176 / 196 = 89.80%
Accuracy: 177 / 197 = 89.85%
Accuracy: 178 / 198 = 89.90%
Accuracy: 179 / 199 = 89.95%
Accuracy: 180 / 200 = 90.00%
Accuracy: 181 / 201 = 90.05%
Accuracy: 182 / 202 = 90.10%
Accuracy: 183 / 203 = 90.15%
Accuracy: 183 / 204 = 89.71%
Accuracy: 184 / 205 = 89.76%


In [6]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
error_log_path = output_path.replace('.txt', '_errors.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(idx, d):
    global acc, total
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth answer

        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Finish your response with: the answer is <answer>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly.\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Get Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
results = []
start_index = 0  # Start processing from question 127
with open(output_path, 'a') as fd, open(bad_output_path, 'a') as bad_fd, open(error_log_path, 'a') as error_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, idx, d) for idx, d in enumerate(dev_data[start_index:], start=start_index)]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                error_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # Final accuracy report
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n")  # Write to the output file


  0%|          | 1/205 [00:10<35:35, 10.47s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/205 [00:13<09:35,  2.87s/it]

Accuracy: 3 / 4 = 75.00%
Accuracy: 4 / 5 = 80.00%
Accuracy: 5 / 6 = 83.33%
Accuracy: 6 / 7 = 85.71%
Accuracy: 7 / 8 = 87.50%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%
Accuracy: 10 / 12 = 83.33%
Accuracy: 10 / 13 = 76.92%
Accuracy: 11 / 14 = 78.57%
Accuracy: 12 / 15 = 80.00%


  8%|▊         | 16/205 [00:17<02:23,  1.31it/s]

...

Accuracy: 169 / 195 = 86.67%


 96%|█████████▌| 196/205 [10:12<00:31,  3.47s/it]

Accuracy: 170 / 196 = 86.73%


100%|██████████| 205/205 [10:20<00:00,  3.03s/it]

Accuracy: 171 / 197 = 86.80%
Accuracy: 172 / 198 = 86.87%
Accuracy: 173 / 199 = 86.93%
Accuracy: 174 / 200 = 87.00%
Accuracy: 175 / 201 = 87.06%
Accuracy: 176 / 202 = 87.13%
Accuracy: 177 / 203 = 87.19%
Accuracy: 177 / 204 = 86.76%
Accuracy: 178 / 205 = 86.83%
